# Risco materno — exploração e treino

Conjunto público **UCI Maternal Health Risk**. A preparação converte temperatura para °C e glicemia para mg/dL, traduz o risco, remove duplicatas e exclui vetores idênticos com rótulos contraditórios antes do split.

A seleção usa validação cruzada estratificada no treino. A métrica principal é o **recall da classe alto**: falso negativo nesse cenário atrasa a revisão do profissional.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from src.data.loader import load_dataset
from src.data.preprocessing import preprocess
from src.ml.train import train_all

raw = load_dataset()
prepared = preprocess(raw)
print("bruto", raw.shape, "preparado", prepared.shape)
print(prepared["risk_level"].value_counts())
prepared.describe().round(1)

In [ ]:
from src.data.preprocessing import FEATURES, prepare_dataset

converted = prepare_dataset(raw).drop_duplicates()
label_count = converted.groupby(FEATURES)["risk_level"].transform("nunique")
print("linhas após duplicatas exatas:", len(converted))
print("grupos de features com rótulos conflitantes:", converted.loc[label_count > 1, FEATURES].drop_duplicates().shape[0])
print("linhas removidas por conflito:", int((label_count > 1).sum()))
print("linhas finais:", len(prepared))

In [ ]:
metrics = train_all()
metrics.round(3)

Depois do treino, `models/cv_metrics.csv` guarda média e desvio dos cinco folds usados na seleção; `models/metrics.csv` contém apenas a avaliação final no teste. `models/importance_*.csv` registra importância global e `models/shap_mean_abs.csv` a média do |SHAP| da random forest.